In [3]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

In [ ]:
from datasets import load_dataset, Dataset, Audio

yue = load_dataset("google/fleurs", "yue_hant_hk", split="train")
en  = load_dataset("google/fleurs", "en_us", split="train")

import pandas as pd
yue_df = yue.to_pandas()[["id", "audio", "transcription"]]
en_df  = en.to_pandas()[["id", "transcription"]].rename(columns={"transcription": "en_text"})

parallel = yue_df.merge(en_df, on="id")
train_dataset = Dataset.from_pandas(parallel)
train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16000))

print(train_dataset)
print(train_dataset[0]["en_text"])


In [ ]:
yue_val = load_dataset("google/fleurs", "yue_hant_hk", split="validation")
en_val  = load_dataset("google/fleurs", "en_us", split="validation")

yue_val_df = yue_val.to_pandas()[["id", "audio", "transcription"]]
en_val_df  = en_val.to_pandas()[["id", "transcription"]].rename(columns={"transcription": "en_text"})

parallel_val = yue_val_df.merge(en_val_df, on="id")
val_dataset = Dataset.from_pandas(parallel_val)
val_dataset = val_dataset.cast_column("audio", Audio(sampling_rate=16000))
print(val_dataset)
print(val_dataset[0]["en_text"])

In [ ]:
from transformers import WhisperFeatureExtractor, WhisperTokenizer

MODEL_NAME = "openai/whisper-small"

feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)
tokenizer = WhisperTokenizer.from_pretrained(
    MODEL_NAME,
    language="chinese",
    task="translate"
)
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(
    MODEL_NAME,
    language="chinese",
    task="translate"
)

In [ ]:
def preprocess(batch):

    audio_arrays = [x["array"] for x in batch["audio"]]
    batch["input_features"] = feature_extractor(
        audio_arrays,
        sampling_rate=16000,
        return_tensors="np"
    ).input_features


    batch["labels"] = tokenizer(batch["en_text"]).input_ids

    return batch

In [ ]:
train_dataset = train_dataset.map(
    preprocess, batched=True, batch_size=8,
    remove_columns=train_dataset.column_names
)
val_dataset = val_dataset.map(
    preprocess, batched=True, batch_size=8,
    remove_columns=val_dataset.column_names
)

In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="chinese",
    task="translate"
)

model.config.forced_decoder_ids = forced_decoder_ids
model.generation_config.forced_decoder_ids = forced_decoder_ids

model.config.suppress_tokens = []
model.generation_config.suppress_tokens = []

model.generation_config.language = "chinese"
model.generation_config.task = "translate"

In [4]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        batch["labels"] = labels
        return batch

NameError: name 'dataclass' is not defined

In [ ]:
!pip install evaluate

In [ ]:
!pip install sacrebleu

In [ ]:
!pip install jiwer

In [ ]:
import evaluate

wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    return {"wer": wer}

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/whisper-yue-en-st",

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,

    learning_rate=1e-5,
    warmup_steps=50,
    max_steps=1000,

    eval_strategy="steps",
    eval_steps=250,

    save_strategy="steps",
    save_steps=250,
    save_total_limit=5,

    logging_steps=25,

    predict_with_generate=True,
    generation_max_length=128,

    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

trainer.train()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls /content/drive/MyDrive/

In [ ]:
!ls /content/drive/MyDrive/whisper-yue-en-st

In [ ]:
best_ckpt = "/content/drive/MyDrive/whisper-yue-en-st/checkpoint-250"
from transformers import WhisperForConditionalGeneration
model = WhisperForConditionalGeneration.from_pretrained(best_ckpt).to("cuda")

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/whisper-yue-en-st",

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,

    learning_rate=1e-5,
    warmup_steps=50,
    max_steps=1000,

    eval_strategy="steps",
    eval_steps=250,

    save_strategy="steps",
    save_steps=250,
    save_total_limit=5,

    logging_steps=25,

    predict_with_generate=True,
    generation_max_length=128,

    fp16=torch.cuda.is_available(),
    report_to="none",
)
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

In [ ]:
checkpoints = [250, 500, 750, 1000]

for ckpt in checkpoints:
    print(f"\n===== checkpoint-{ckpt} =====")

    model = WhisperForConditionalGeneration.from_pretrained(
        f"/content/drive/MyDrive/whisper-yue-en-st/checkpoint-{ckpt}"
    ).to("cuda")

    forced_decoder_ids = processor.get_decoder_prompt_ids(
        language="chinese",
        task="translate"
    )

    model.generation_config.forced_decoder_ids = forced_decoder_ids
    model.generation_config.language = "chinese"
    model.generation_config.task = "translate"
    model.generation_config.suppress_tokens = []

    test_metrics = trainer.evaluate(eval_dataset=test_dataset)
    print(test_metrics)

    test_predictions = trainer.predict(test_dataset)

In [5]:
import evaluate

wer_metric = evaluate.load("wer")
bleu_metric = evaluate.load("sacrebleu")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    bleu = bleu_metric.compute(
        predictions=pred_str,
        references=[[x] for x in label_str]
    )["score"]

    return {
        "wer": wer,
        "bleu": bleu
    }

In [6]:
trainer.compute_metrics = compute_metrics

NameError: name 'trainer' is not defined

In [ ]:
metrics = trainer.evaluate()
print(metrics)

In [ ]:
checkpoints = [250, 500, 750, 1000]

for ckpt in checkpoints:
    print(f"\n===== checkpoint-{ckpt} =====")

    model = WhisperForConditionalGeneration.from_pretrained(
        f"/content/drive/MyDrive/whisper-yue-en-st/checkpoint-{ckpt}"
    ).to("cuda")

    forced_decoder_ids = processor.get_decoder_prompt_ids(
        language="chinese",
        task="translate"
    )

    model.generation_config.forced_decoder_ids = forced_decoder_ids
    model.generation_config.language = "chinese"
    model.generation_config.task = "translate"

    trainer.model = model

    metrics = trainer.evaluate()
    print(metrics)

In [ ]:
import pandas as pd

results = pd.DataFrame({
    "checkpoint": [250, 500, 750, 1000],
    "eval_loss": [2.7212648391723633, 2.8511531352996826, 2.9221842288970947, 3.0427122116088867],
    "wer": [92.71062458437773, 96.70571384725561, 93.09427592204204, 97.30932528518082],
    "bleu": [5.951021268853995, 5.020203205896263, 5.028413770063452, 5.0000762462968]
})

results.to_csv("/content/drive/MyDrive/whisper-yue-en-st/evaluation_results.csv", index=False)

print(results)

In [ ]:
import pandas as pd

results = pd.DataFrame({
    "checkpoint": [250, 500, 750, 1000],
    "eval_loss": [2.7212648391723633, 2.8511531352996826, 2.9221842288970947, 3.0427122116088867],
    "wer": [92.71062458437773, 96.70571384725561, 93.09427592204204, 97.30932528518082],
    "bleu": [5.951021268853995, 5.020203205896263, 5.028413770063452, 5.0000762462968]
})

results.to_csv("/content/drive/MyDrive/whisper-yue-en-st/evaluation_results.csv", index=False)

print(results)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

save_dir = "/content/drive/MyDrive/whisper-yue-en-st/figures"
os.makedirs(save_dir, exist_ok=True)

df = pd.DataFrame({
    "checkpoint": [250, 500, 750, 1000],
    "train_loss": [1.95, 0.92, 0.78, 0.50],
    "val_loss": [2.721, 2.851, 2.922, 3.043],
    "wer": [92.71, 96.71, 93.09, 97.31],
    "bleu": [5.95, 5.02, 5.03, 5.00]
})

df

In [ ]:
plt.figure(figsize=(7, 4.5))

plt.plot(df["checkpoint"], df["train_loss"], marker="o", label="Training Loss")
plt.plot(df["checkpoint"], df["val_loss"], marker="o", label="Validation Loss")

plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.xticks(df["checkpoint"])
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{save_dir}/loss_curve.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(7, 4.5))

plt.plot(
    df["checkpoint"],
    df["wer"],
    marker="o",
    linewidth=2,
    label="WER"
)

plt.xlabel("Checkpoint")
plt.ylabel("WER (%)")
plt.title("WER across Checkpoints")

plt.xticks(df["checkpoint"])
plt.ylim(80, 100)

plt.grid(True, alpha=0.3)

best_idx = df["wer"].idxmin()

plt.scatter(
    df.loc[best_idx, "checkpoint"],
    df.loc[best_idx, "wer"],
    s=70,
    label="Best checkpoint"
)

plt.legend(loc="upper right")

plt.tight_layout()
plt.savefig(f"{save_dir}/wer_curve.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(7, 4.5))

plt.plot(
    df["checkpoint"],
    df["bleu"],
    marker="o",
    linewidth=2,
    label="BLEU"
)

plt.xlabel("Checkpoint")
plt.ylabel("BLEU")
plt.title("BLEU across Checkpoints")
plt.xticks(df["checkpoint"])
plt.ylim(5, 10)
plt.grid(True, alpha=0.3)

best_idx = df["bleu"].idxmax()

plt.scatter(
    df.loc[best_idx, "checkpoint"],
    df.loc[best_idx, "bleu"],
    s=70,
    label="Best checkpoint"
)

plt.legend(loc="upper right")

plt.tight_layout()
plt.savefig(f"{save_dir}/bleu_curve.png", dpi=300)
plt.show()

In [ ]:
fig, ax1 = plt.subplots(figsize=(7, 4.5))

# WER
line1 = ax1.plot(
    df["checkpoint"],
    df["wer"],
    marker="o",
    linewidth=2,
    label="WER"
)

ax1.set_xlabel("Checkpoint")
ax1.set_ylabel("WER (%)")
ax1.set_xticks(df["checkpoint"])
ax1.set_ylim(85, 100)
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()

line2 = ax2.plot(
    df["checkpoint"],
    df["bleu"],
    marker="s",
    linewidth=2,
    linestyle="--",
    label="BLEU"
)

ax2.set_ylabel("BLEU")
ax2.set_ylim(5, 10)

lines = line1 + line2
labels = [l.get_label() for l in lines]

ax1.legend(lines, labels, loc="upper right")

plt.title("Translation Performance across Checkpoints")

fig.tight_layout()
plt.savefig(f"{save_dir}/wer_bleu_curve.png", dpi=300)
plt.show()

In [ ]:
table_df = df[["checkpoint", "val_loss", "wer", "bleu"]].copy()
table_df.columns = ["Checkpoint", "Validation Loss ↓", "WER ↓", "BLEU ↑"]

fig, ax = plt.subplots(figsize=(7, 2.2))
ax.axis("off")

table = ax.table(
    cellText=table_df.round(3).values,
    colLabels=table_df.columns,
    cellLoc="center",
    loc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.5)

plt.title("Checkpoint Evaluation Results", pad=15)

plt.tight_layout()
plt.savefig(f"{save_dir}/results_table.png", dpi=300)
plt.show()

In [ ]:
dataset_df = pd.DataFrame({
    "Split": ["Train", "Validation"],
    "Examples": [3385, "1900+"],
    "Input": ["Cantonese speech", "Cantonese speech"],
    "Target": ["English text", "English text"]
})

fig, ax = plt.subplots(figsize=(7, 2))
ax.axis("off")

table = ax.table(
    cellText=dataset_df.values,
    colLabels=dataset_df.columns,
    cellLoc="center",
    loc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.5)

plt.title("Dataset Summary", pad=15)

plt.tight_layout()
plt.savefig(f"{save_dir}/dataset_table.png", dpi=300)
plt.show()